# main_analysis.py
Converted to a notebook with separate cells per function/class.

## 0) Setup & autoreload

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("/Users/yanasklar/GitHub/Enjoyment")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
print("Added to sys.path:", PROJECT_ROOT)

%load_ext autoreload
%autoreload 2

!python --version

## 1) Imports

In [ ]:
import logging
import os
import pickle
from pathlib import Path
from typing import Dict, Any
from logger_config import configure_logging
from participant_data import *
from questionnaire_loader import load_questionnaire_data
from visualizations import *
import pandas as pd
from scipy.stats import spearmanr
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from scipy.stats import kruskal
import scikit_posthocs as sp
from matplotlib.patches import Patch

## 2) Top-level configuration / constants

In [ ]:
# Save
    stats_df.to_csv("Questionnaire_Descriptive_Stats.csv", index=True)
    print(f"✅ Descriptive statistics saved")

    return stats_df
        if questionnaire is None:
            continue

        for painting in ["Klimt", "Pollock", "van Dongen", "Braque", "de Chirico", "Janco", "Picasso"]:
            rating = questionnaire.get(painting)
            if pd.notna(rating):
                data.append({
                    "ParticipantID": pid,
                    "TourType": tour_type,
                    "Rating": int(rating)
                })

    return pd.DataFrame(data)
    # Compute average rating per tour type
    avg_by_type = df.groupby("TourType")["Rating"].mean().round(2).to_dict()

    # Compute proportions
    counts = df.groupby(["TourType", "RatingLabel"]).size().reset_index(name="Count")
    total_per_type = df["TourType"].value_counts().to_dict()
    counts["Proportion"] = counts.apply(lambda row: row["Count"] / total_per_type[row["TourType"]], axis=1)

    # Define consistent custom colors for the 3 tour types
    custom_palette = {
        1: "#3C3C3C",   # Dark gray
        2: "#B76E79",   # Dusty pink
        3: "#D9C8B4"    # Light beige
    }

    # Plot
    plt.figure(figsize=(14, 8))
    sns.barplot(
        data=counts,
        x="RatingLabel",
        y="Proportion",
        hue="TourType",
        palette=custom_palette
    )

    legend_patches = [
        Patch(color=custom_palette[t], label=f"{t} {'Fully active' if t==1 else 'Semi-active' if t==2 else 'Passive'} (Avg: {avg_by_type[t]})")
        for t in sorted(avg_by_type.keys())
    ]
    plt.legend(title="Type", handles=legend_patches)

    plt.title("How much you liked all the paintings from the museum by Type")
    plt.xlabel("Rating")
    plt.ylabel("Proportion")
    plt.xticks(rotation=45)
    plt.ylim(0, 0.3)
    plt.tight_layout()
    plt.show()
    plt.savefig("Average Liking by Type.png")

def analyze_liking_vs_preference(questionnaire_df: pd.DataFrame, participants: dict) -> pd.DataFrame:
    """
    Correlates how much participants liked each artwork with how often they preferred it over a new artwork.
    Calculates results overall and split by TourType.

    Returns:
        DataFrame with columns: Painting, TourType, SpearmanRho, p-value, N
    """

    # Define the mapping of paintings to liking column and preference column + which option they are
    painting_map = {
        "Klimt":       {"liking_col": "Klimt", "pref_col": "Preferred image",       "option": 1},
        "van Dongen":  {"liking_col": "van Dongen", "pref_col": "Preferred image.1",     "option": 2},
        "Braque":      {"liking_col": "Braque", "pref_col": "Preferred image.2",     "option": 1},
        "Pollock":     {"liking_col": "Pollock", "pref_col": "Preferred image.3", "option": 2},
        "de Chirico":  {"liking_col": "de Chirico", "pref_col": "Preferred image.4",     "option": 2},
        "Janco":       {"liking_col": "Janco", "pref_col": "Preferred image.5",     "option": 2},
        "Picasso":     {"liking_col": "Picasso", "pref_col": "Preferred image.6",     "option": 2},
    }

    questionnaire_df["ParticipantID"] = questionnaire_df.index
    
    # Extract TourType per participant
    tour_type_map = {int(pid): p.tour_type for pid, p in participants.items()}
    questionnaire_df["TourType"] = questionnaire_df["ParticipantID"].map(tour_type_map)

    results = []

    for painting, config in painting_map.items():
        liking_col = config["liking_col"]
        pref_col = config["pref_col"]
        correct_option = str(config["option"])

        # Create binary preference: 1 if chosen, 0 if not
        questionnaire_df[f"{painting}_Preferred"] = (
            questionnaire_df[pref_col].astype(str) == correct_option
        ).astype(int)
        # Run correlation overall and by TourType
        for tour in ["All", 1, 2, 3]: 
            if tour == "All":
                subset = questionnaire_df
                label = "All"
            else:
                subset = questionnaire_df[questionnaire_df["TourType"] == tour]
                label = f"Type type: {tour}"

            x = pd.to_numeric(subset[liking_col], errors="coerce")
            #print("x = ", x)
            if f"{painting}_Preferred" not in subset.columns:
                logging.info(f"{painting}_Preferred column not found. Skipping.")
                continue
            y = pd.to_numeric(subset[f"{painting}_Preferred"], errors="coerce")
            #print("y = ", y)

            valid = (~x.isna()) & (~y.isna())
            x, y = x[valid], y[valid]

            if len(x) > 2:
                rho, p = spearmanr(x, y)
                results.append({
                    "Painting": painting,
                    "TourType": label,
                    "SpearmanRho": round(rho, 3),
                    "p-value": round(p, 4),
                    "N": len(x)
                })
    results = pd.DataFrame(results)
    results.to_csv("Liking_vs_Preference_by_TourType.csv", index=False)
    return pd.DataFrame(results)
    correlations_painting = []
    for col in numeric_cols_painting:
        if col == target:
            continue
        rho, p = spearmanr(subset_painting[target], subset_painting[col])
        correlations_painting.append({
            "Feature": col,
            "SpearmanRho": round(rho, 3),
            "p_value": round(p, 4)
        })

    df_painting = pd.DataFrame(correlations_painting).sort_values(by="SpearmanRho", key=abs, ascending=False)
    df_painting.to_csv("correlation_of_VR_measures_with_liking_per_painting.csv", index=False)

    # --- Participant-Level Analysis ---
    target_2 = "AvgGeneralRating"
    numeric_cols_participant = per_Participant_Summary.select_dtypes(include='number').columns.tolist()
    numeric_cols_participant = [col for col in numeric_cols_participant if col != "ParticipantID"]
    subset_participant = per_Participant_Summary[numeric_cols_participant].dropna()

    if target_2 not in subset_participant.columns:
        print(f"⚠️ '{target_2}' not found in per_Participant_Summary. Skipping participant-level analysis.")
        return

    correlations_participant = []
    for col in numeric_cols_participant:
        if col == target_2:
            continue
        rho, p = spearmanr(subset_participant[target_2], subset_participant[col])
        correlations_participant.append({
            "Feature": col,
            "SpearmanRho": round(rho, 3),
            "p_value": round(p, 4)
        })

    df_participant = pd.DataFrame(correlations_participant).sort_values(by="SpearmanRho", key=abs, ascending=False)
    df_participant.to_csv("correlation_of_VR_measures_with_liking_per_participant.csv", index=False)
    print("✅ Saved correlation results to both correlation_of_VR_measures_with_liking_per_painting.csv and _per_participant.csv")
    results = []
    for col in question_columns:
        groups = [questionnaire[questionnaire["Type"] == t][col].dropna() for t in [1, 2, 3]]
        stat, p = kruskal(*groups)
        results.append({
            "Question": col,
            "H-stat": round(stat, 3),
            "p-value": round(p, 4)
        })

    pd.DataFrame(results).to_csv("Questionnaire_Kruskal_by_Type.csv", index=False)
    results = []
    for metric in grouped.mean().columns:
        groups = [per_Participant_Summary[per_Participant_Summary["TourType"] == t][metric].dropna() for t in [1, 2, 3]]
        stat, p = kruskal(*groups)
        results.append({"Metric": metric, "H-stat": round(stat, 3), "p-value": round(p, 4)})

    pd.DataFrame(results).sort_values("p-value")\
        .to_csv("Measures per participant across types, kruskal.csv", index=False)
    grouped = merged.groupby("TourType")[[
        "Audio_AvgValence", "FI_Valence", "ReactionTime", 
        "GazePercent_Audio", "SaccadeRate", 
        "Audio_DominantEmotionIntensity", "FI_MaxIntensity", "GazeTime"
    ]]
    grouped.agg(["mean", "std", "median", "min", "max", "count"]).round(2)\
        .to_csv("Painting_Level_Descriptive_Stats.csv")

    results = []
    for metric in grouped.mean().columns:
        groups = [merged[merged["TourType"] == t][metric].dropna() for t in [1, 2, 3]]
        stat, p = kruskal(*groups)
        results.append({"Metric": metric, "H-stat": round(stat, 3), "p-value": round(p, 4)})

    pd.DataFrame(results).sort_values("p-value")\
        .to_csv("Measures per painting across types, kruskal.csv", index=False)
    
    return merged  # reused later
    correlations = []
    for metric in metrics:
        for t in [1, 2, 3]:
            subset = merged_df[merged_df["TourType"] == t]
            x = subset[metric]
            y = subset["SelfReportedLiking"]
            rho, p = spearmanr(x, y, nan_policy='omit')
            correlations.append({
                "Metric": metric, "TourType": t, "SpearmanRho": round(rho, 3), "p-value": round(p, 4)
            })

    pd.DataFrame(correlations).sort_values("p-value")\
        .to_csv("Measures per painting across types, spearman.csv", index=False)
    correlations = []
    for metric in metrics:
        for t in [1, 2, 3]:
            subset = per_Participant_Summary[per_Participant_Summary["TourType"] == t]
            x = subset[metric]
            y = subset["AvgGeneralRating"]
            rho, p = spearmanr(x, y, nan_policy='omit')
            correlations.append({
                "Metric": metric,
                "TourType": t,
                "SpearmanRho": round(rho, 3),
                "p-value": round(p, 4)
            })

    pd.DataFrame(correlations).sort_values("p-value")\
        .to_csv("Measures per participant across types, spearman.csv", index=False)

## 3) Functions & Classes (one per cell)

In [ ]:

def compute_questionnaire_descriptive_stats(questionnaire_df: pd.DataFrame):
    """
    Computes descriptive statistics (mean, std, median, min, max, count) for all numerical columns in the questionnaire.
    Saves the result as a CSV.
    """
    # Ensure numeric columns only
    numeric_df = questionnaire_df.select_dtypes(include='number')
    
    # Compute descriptive statistics
    stats_df = numeric_df.agg(['mean', 'std', 'median', 'min', 'max']).T.round(2)
    stats_df.index.name = "Question"

    # Save
    stats_df.to_csv("Questionnaire_Descriptive_Stats.csv", index=True)
    print(f"✅ Descriptive statistics saved")
    print(stats_df)
    return stats_df


In [ ]:

def collect_all_ratings(participants):
    data = []
    for participant in participants.values():
        pid = int(participant.participant_id)
        tour_type = participant.tour_type
        questionnaire = participant.questionnaire_data


In [ ]:

def plot_rating_distribution(df):
    # Map rating numbers to descriptive labels
    rating_labels = {
        1: "Did not like at all", 2: "Did not like", 3: "Quite disliked", 4: "Neutral",
        5: "Quite liked", 6: "Liked", 7: "Really liked"
    }
    df["RatingLabel"] = df["Rating"].map(rating_labels)


In [1]:

def analyze_vr_correlation_with_liking(per_Painting_Summary: pd.DataFrame, per_Participant_Summary: pd.DataFrame):
    # --- Painting-Level Analysis ---
    target = "SelfReportedLiking"
    numeric_cols_painting = per_Painting_Summary.select_dtypes(include='number').columns.tolist()
    numeric_cols_painting = [col for col in numeric_cols_painting if col != "ParticipantID"]
    subset_painting = per_Painting_Summary[numeric_cols_painting].dropna()

    correlations_painting = []
    for col in numeric_cols_painting:
        if col == target:
            continue
        rho, p = spearmanr(subset_painting[target], subset_painting[col])
        correlations_painting.append({
            "Feature": col,
            "SpearmanRho": round(rho, 3),
            "p_value": round(p, 4)
        })

    df_painting = pd.DataFrame(correlations_painting).sort_values(by="SpearmanRho", key=abs, ascending=False)
    df_painting.to_csv("correlation_of_VR_measures_with_liking_per_painting.csv", index=False)

    # --- Participant-Level Analysis ---
    target_2 = "AvgGeneralRating"
    numeric_cols_participant = per_Participant_Summary.select_dtypes(include='number').columns.tolist()
    numeric_cols_participant = [col for col in numeric_cols_participant if col != "ParticipantID"]
    subset_participant = per_Participant_Summary[numeric_cols_participant].dropna()

    if target_2 not in subset_participant.columns:
        print(f"⚠️ '{target_2}' not found in per_Participant_Summary. Skipping participant-level analysis.")
        return

    correlations_participant = []
    for col in numeric_cols_participant:
        if col == target_2:
            continue
        rho, p = spearmanr(subset_participant[target_2], subset_participant[col])
        correlations_participant.append({
            "Feature": col,
            "SpearmanRho": round(rho, 3),
            "p_value": round(p, 4)
        })

    df_participant = pd.DataFrame(correlations_participant).sort_values(by="SpearmanRho", key=abs, ascending=False)
    df_participant.to_csv("correlation_of_VR_measures_with_liking_per_participant.csv", index=False)
    print("✅ Saved correlation results to both correlation_of_VR_measures_with_liking_per_painting.csv and _per_participant.csv")
    print(df_participants)

NameError: name 'pd' is not defined

In [ ]:

def run_questionnaire_anova(questionnaire_path: str):
    questionnaire = pd.read_csv(questionnaire_path, sep=";", quoting=3).drop(columns=["Timestamp"])
    questionnaire = questionnaire.rename(columns={"Number": "ParticipantID"})
    questionnaire["Type"] = questionnaire["Type"].str.extract(r"(\d)").astype(int)
    question_columns = questionnaire.columns[2:16]


In [ ]:

def run_per_participant_kruskal(per_Participant_Summary: pd.DataFrame):
    grouped = per_Participant_Summary.groupby("TourType")[[
        "AvgGazeTime", "AvgGazePercent_Audio", "AvgFI_Valence", "AvgAudio_Valence",
        "AvgAudio_DominantIntensity", "AvgSaccadeRate", "TotalExperimentTime", "AvgSpeed"
    ]]
    grouped.agg(["mean", "std", "median", "min", "max", "count"]).round(2)\
        .to_csv("Participant_Level_Descriptive_Stats.csv")


In [ ]:

def run_per_painting_kruskal(per_Painting_Summary: pd.DataFrame, per_Participant_Summary: pd.DataFrame):
    participant_df = per_Participant_Summary[["ParticipantID", "TourType"]]
    merged = per_Painting_Summary.merge(participant_df, on="ParticipantID")


In [ ]:

def run_per_painting_spearman(merged_df: pd.DataFrame):
    metrics = [
        "FI_Valence", "FI_MaxIntensity", "GazeTime", "GazePercent_Audio", 
        "Audio_AvgValence", "Audio_DominantEmotionIntensity", "SaccadeRate", "ReactionTime"
    ]


In [ ]:

def run_per_participant_spearman(per_Participant_Summary: pd.DataFrame):
    metrics = [
        "AvgGazeTime", "AvgGazePercent_Audio", "AvgFI_Valence", "AvgAudio_Valence",
        "AvgAudio_DominantIntensity", "AvgSaccadeRate", "TotalExperimentTime", "AvgSpeed"
    ]


## 5) Save outputs helper

In [ ]:
from pathlib import Path
outdir = Path("/mnt/data/outputs"); outdir.mkdir(exist_ok=True, parents=True)
print("Saving to:", outdir)